# FlyBot Sentinel: fruit-fly detector (YOLO11n) on a Colab T4

Trains **YOLO11n** (1 class `fruit_fly`) on the CamOS Roboflow dataset
[camos/fruit-fly-wmcqi v3](https://universe.roboflow.com/camos/fruit-fly-wmcqi/dataset/3)
(**CC BY 4.0**, YOLOv8 export `fruit-fly.v3i.yolov8.zip`), evaluates on the 76-image test split,
exports **ONNX opset 17, 640x640 static, simplified** for OpenCV 5 DNN, and downloads the results.

Same settings as `models/train.py` in the flybot-sentinel repo (T1.3). Nothing private is cloned.

1. Runtime > Change runtime type > **T4 GPU**
2. Run all. The dataset cell asks for the zip (or put it at `DATASET_ZIP` first, e.g. on Drive).

In [ ]:
# Settings
MODEL = 'yolo11n.pt'          # fallback: 'yolov8n.pt'
EPOCHS = 80                   # 60-100 planned for T1.3
IMGSZ = 640
BATCH = 32
NAME = 'fruitfly-yolo11n'
DATASET_ZIP = '/content/fruit-fly.v3i.yolov8.zip'
SMOKE_TEST = False            # True: 1 epoch on 5% of train, to check the notebook end to end
ULTRALYTICS = '8.4.177'       # same version as models/uv.lock

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics=={ULTRALYTICS} onnx onnxslim

In [ ]:
# Dataset: upload the zip if it is not there, unzip, rewrite polygon rows as boxes, absolute data.yaml
import os
import shutil
from pathlib import Path

import yaml

if not os.path.exists(DATASET_ZIP):
    from google.colab import files

    uploaded = files.upload()  # pick fruit-fly.v3i.yolov8.zip
    DATASET_ZIP = '/content/' + next(iter(uploaded))
DATA = Path('/content/data')
if not (DATA / 'train').exists():
    shutil.unpack_archive(DATASET_ZIP, DATA)


def polygon_rows_to_boxes(label_dir):
    # 31 train files hold polygon rows, 3 of them mixed with box rows (ultralytics drops those as corrupt)
    rows = 0
    for txt in Path(label_dir).glob('*.txt'):
        out, changed = [], False
        for line in txt.read_text().splitlines():
            p = line.split()
            if len(p) >= 7 and len(p) % 2 == 1:
                xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
                x0, x1, y0, y1 = max(0, min(xs)), min(1, max(xs)), max(0, min(ys)), min(1, max(ys))
                line = f'{p[0]} {(x0 + x1) / 2:.6f} {(y0 + y1) / 2:.6f} {x1 - x0:.6f} {y1 - y0:.6f}'
                rows, changed = rows + 1, True
            out.append(line)
        if changed:
            txt.write_text('\n'.join(out) + '\n')
    return rows


for split in ['train', 'valid', 'test']:
    n = len(list((DATA / split / 'images').glob('*')))
    print(split, n, 'images,', polygon_rows_to_boxes(DATA / split / 'labels'), 'polygon rows -> boxes')
data_yaml = {'path': str(DATA), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
             'nc': 1, 'names': ['fruit_fly']}
Path('/content/data.yaml').write_text(yaml.safe_dump(data_yaml, sort_keys=False))

In [ ]:
# Train (ultralytics prints the per-epoch time: multiply by EPOCHS for the ETA)
from ultralytics import YOLO, settings

settings.update({'sync': False})
model = YOLO(MODEL)
model.train(data='/content/data.yaml', epochs=1 if SMOKE_TEST else EPOCHS, imgsz=IMGSZ, batch=BATCH,
            fraction=0.05 if SMOKE_TEST else 1.0, device=0, workers=4, patience=30, seed=0,
            deterministic=True, cache='ram', project='/content/runs', name=NAME, exist_ok=True)
RUN = Path(model.trainer.save_dir)
BEST = RUN / 'weights' / 'best.pt'

In [ ]:
# Test split (76 images): the T1.3 acceptance number is mAP50 here
import json

r = YOLO(str(BEST)).val(data='/content/data.yaml', split='test', imgsz=IMGSZ, batch=16, device=0,
                        project=str(RUN), name='test', exist_ok=True)
metrics = {'split': 'test', 'imgsz': IMGSZ, 'mAP50': round(float(r.box.map50), 4),
           'mAP50_95': round(float(r.box.map), 4), 'precision': round(float(r.box.mp), 4),
           'recall': round(float(r.box.mr), 4), 'smoke_run': SMOKE_TEST, 'epochs': EPOCHS}
(RUN / 'test_metrics.json').write_text(json.dumps(metrics, indent=2))
print(metrics)

In [ ]:
# ONNX for OpenCV 5 DNN: opset 17, 640 static, simplified, no NMS in the graph (+ a dynamic copy for T1.4)
onnx_static = YOLO(str(BEST)).export(format='onnx', opset=17, imgsz=640, dynamic=False, simplify=True,
                                     nms=False, batch=1, device='cpu')
shutil.copy(onnx_static, f'/content/{NAME}.onnx')
onnx_dynamic = YOLO(str(BEST)).export(format='onnx', opset=17, imgsz=640, dynamic=True, simplify=True,
                                      nms=False, device='cpu')
shutil.copy(onnx_dynamic, f'/content/{NAME}-dynamic.onnx')

In [ ]:
# Download: ONNX files, results.csv, test metrics, and the run folder (weights, plots)
from google.colab import files

shutil.copy(RUN / 'results.csv', f'/content/{NAME}-results.csv')
shutil.copy(RUN / 'test_metrics.json', f'/content/{NAME}-test_metrics.json')
shutil.make_archive(f'/content/{NAME}-run', 'zip', RUN)
for f in [f'{NAME}.onnx', f'{NAME}-dynamic.onnx', f'{NAME}-results.csv', f'{NAME}-test_metrics.json',
          f'{NAME}-run.zip']:
    files.download('/content/' + f)